# Performance Relation

分析原始数据序列与创造性绩效的关系。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

import sys
sys.path.insert(0, str(Path('../../../modules').resolve()))
from pipeline_config import *

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 输入文件路径（从配置读取）
output_dir = BY_RAW_OUTPUT
raw_sequences_file = output_dir / 'raw_sequences.csv'
performance_file = PERFORMANCE_FILE

In [ ]:
# 加载数据
df_raw_sequences = pd.read_csv(raw_sequences_file)
df_performance = pd.read_csv(performance_file)

In [ ]:
# 1. 计算原始序列余弦相似性矩阵
from sklearn.metrics.pairwise import cosine_similarity
import ast
import json

def cosine_similarity_matrix(df_sequences):
    # 解析字符串形式的数组，将其转换回numpy array
    def parse_sequence(seq_str):
        if not isinstance(seq_str, str):
            return seq_str
        try:
            return json.loads(seq_str)
        except:
            pass
        try:
            return ast.literal_eval(seq_str)
        except:
            # 备用方案：处理可能是无逗号空格分隔的数组字符串
            clean_str = seq_str.strip('[]\n\r ')
            if ',' in clean_str:
                return [float(x) for x in clean_str.split(',')]
            return [float(x) for x in clean_str.split()]
            
    # 解析并构成特征矩阵 X
    parsed_seqs = df_sequences['raw_sequence'].apply(parse_sequence).tolist()
    X = np.array(parsed_seqs)
    
    # 用sklearn计算两两序列的余弦相似度矩阵
    cos_sim_matrix = cosine_similarity(X)
    return cos_sim_matrix

In [ ]:
# 2. 计算表现之间的差值的绝对值矩阵

def performance_difference_matrix(performance_df, key):
    # 提取表现得分
    perf_scores = performance_df[key].values
    
    # 利用numpy的广播机制，直接计算两两之间分数的差值绝对值
    # perf_scores[:, None] 将 (N,) 变成 (N, 1)
    # perf_scores[None, :] 变成 (1, N)
    diff_matrix = np.abs(perf_scores[:, None] - perf_scores[None, :])
    
    return diff_matrix

In [ ]:
# 3. 计算两个矩阵之间的相关性
from scipy.stats import pearsonr, spearmanr

def calculate_correlation(matrix1, matrix2, method='spearman'):
    # 取矩阵的上三角（不含对角线 k=1）的部分进行相关性计算
    # 这样可以避免自身与自身的相关（对角线），以及对称性的冗余（双倍计算）
    indices = np.triu_indices_from(matrix1, k=1)
    
    vec1 = matrix1[indices]
    vec2 = matrix2[indices]
    
    # 计算相关系数，通常用Spearman评估非线性单调关系，因为表现差距不一定是线性的
    if method == 'pearson':
        corr, pval = pearsonr(vec1, vec2)
    else:
        corr, pval = spearmanr(vec1, vec2)
        
    return corr

In [ ]:
# 4. 验证相关性的可靠性，进行以下迭代1000~10000次：
# (1) 随机打乱原创性矩阵的数据点分布
# (2) 重新计算相关性
# (3) 记录每次迭代的相关性值，形成一个分布
# (4) 计算原始相关性在这个分布中的位置，评估其显著性
from tqdm import tqdm

def shuffle_matrix(matrix):
    # Mantel test要求：在打乱矩阵时，必须同步打乱行和列的索引。
    # 这样才能保持矩阵内部的结构逻辑假说被打破，同时仍保留原有的节点度分布。
    n = matrix.shape[0]
    idx = np.random.permutation(n)
    # matrix[idx, :] 会打乱行，接着 [:, idx] 会按照同样的顺序打乱列
    return matrix[idx, :][:, idx]

def permutation_test(cosine_matrix, originality_matrix, num_iterations=1000, method='spearman'):
    # 首先计算实际观察到的真实相关性
    true_corr = calculate_correlation(cosine_matrix, originality_matrix, method=method)
    
    null_distribution = np.zeros(num_iterations)
    
    # 不断打乱和重算，构造零假设分布
    for i in tqdm(range(num_iterations), desc="Permutation Testing"):
        shuffled_orig = shuffle_matrix(originality_matrix)
        null_distribution[i] = calculate_correlation(cosine_matrix, shuffled_orig, method=method)
        
    # 计算双尾回归 P 值：记录比观察到的真实相关性“更极端”的数据占比
    p_value = np.sum(np.abs(null_distribution) >= np.abs(true_corr)) / num_iterations
    
    return true_corr, p_value, null_distribution

In [ ]:
# 主流程

# 计算余弦相似性矩阵
cosine_matrix = cosine_similarity_matrix(df_raw_sequences)

# 计算表现差值矩阵
performance_matrix = performance_difference_matrix(df_performance, 'fluency')

# 计算相关性
correlation, p_value, null_dist = permutation_test(cosine_matrix, performance_matrix, num_iterations=1000, method='spearman')

print(f"Observed Spearman Correlation: {correlation:.4f}")
print(f"P-value from Permutation Test: {p_value:.4f}")

In [ ]:
# 可视化零假设分布和观察到的相关性
plt.figure(figsize=(10, 6))
plt.hist(null_dist, bins=50, alpha=0.7, label='Null Distribution')
plt.axvline(correlation, color='red', linestyle='--', linewidth=2, label=f'Observed Correlation: {correlation:.4f}')
plt.xlabel('Spearman Correlation')
plt.ylabel('Frequency')
plt.title('Permutation Test for Performance Relation')
plt.legend()
plt.show()

## 分析流程可视化

展示 Mantel 检验的核心数据结构：余弦相似性矩阵、绩效差异矩阵、以及两者上三角值的关系。
此图依赖前面主流程已计算的 `cosine_matrix` 和 `performance_matrix`（流畅性），无需重新运行。

In [ ]:
# ============================================================
# 分析流程可视化：展示 Mantel 检验的核心数据结构
# ============================================================
from matplotlib import rcParams
rcParams['font.size'] = 13

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# --- 子图1: 余弦相似性矩阵（前60名被试）---
n_show = 60
ax = axes[0]
im = ax.imshow(cosine_matrix[:n_show, :n_show], cmap='RdYlBu_r', aspect='auto')
ax.set_title('A. AI调用序列余弦相似性矩阵', fontsize=14, fontweight='bold')
ax.set_xlabel('被试编号', fontsize=12)
ax.set_ylabel('被试编号', fontsize=12)
cbar = plt.colorbar(im, ax=ax, shrink=0.75)
cbar.set_label('余弦相似度', fontsize=11)

# --- 子图2: 流畅性差异矩阵（前60名被试）---
ax = axes[1]
im = ax.imshow(performance_matrix[:n_show, :n_show], cmap='YlOrRd', aspect='auto')
ax.set_title('B. 流畅性差异矩阵', fontsize=14, fontweight='bold')
ax.set_xlabel('被试编号', fontsize=12)
ax.set_ylabel('被试编号', fontsize=12)
cbar = plt.colorbar(im, ax=ax, shrink=0.75)
cbar.set_label('|流畅性差值|', fontsize=11)

# --- 子图3: 上三角值散点图 ---
indices = np.triu_indices_from(cosine_matrix, k=1)
vec_cos = cosine_matrix[indices]
vec_perf = performance_matrix[indices]
# 随机采样以加速绘制
n_sample = min(8000, len(vec_cos))
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(vec_cos), n_sample, replace=False)

ax = axes[2]
ax.scatter(vec_cos[sample_idx], vec_perf[sample_idx], alpha=0.25, s=4,
           c='#4472C4', edgecolors='none')
ax.set_xlabel('序列余弦相似度', fontsize=12)
ax.set_ylabel('流畅性绝对差值', fontsize=12)
ax.set_title('C. 矩阵上三角关系\n(随机采样点)', fontsize=14, fontweight='bold')

plt.tight_layout(pad=1.5)
plt.show()

## 流畅性 (Fluency) — Mantel 置换检验

优化版可视化：更大的中文字号、更清晰的图例。数据复用前面主流程的 `correlation` 和 `null_dist`。

In [ ]:
# ============================================================
# 流畅性 — 置换检验零假设分布（优化版）
# ============================================================
plt.figure(figsize=(9, 5.5))
plt.hist(null_dist, bins=45, alpha=0.75, color='#5B9BD5', edgecolor='white', linewidth=0.5,
         label='零假设分布')
plt.axvline(correlation, color='#C00000', linestyle='--', linewidth=2.5,
           label=f'观测值 ρ = {correlation:.4f}')
plt.axvline(0, color='gray', linestyle=':', linewidth=1)

plt.xlabel('Spearman 相关系数', fontsize=15)
plt.ylabel('频次', fontsize=15)
plt.title('流畅性 (Fluency) — Mantel置换检验', fontsize=17, fontweight='bold')
plt.legend(fontsize=13, loc='upper left')
plt.tick_params(labelsize=12)
plt.tight_layout()
plt.show()

## 原创性 (Originality) — Mantel 置换检验

对原创性（Ocsai 评分）执行与流畅性相同的 Mantel 检验流程，检验 AI 调用序列相似性与原创性绩效相似性之间的关联。

In [ ]:
# ============================================================
# 原创性 — Mantel检验
# ============================================================

# 计算原创性差异矩阵（复用前面定义的 performance_difference_matrix 函数）
orig_matrix = performance_difference_matrix(df_performance, 'originality')

# 置换检验（1000次迭代）
corr_orig, p_orig, null_orig = permutation_test(
    cosine_matrix, orig_matrix,
    num_iterations=1000, method='spearman'
)

print(f"原创性 Mantel 检验结果:")
print(f"  观测 Spearman ρ = {corr_orig:.4f}")
print(f"  置换检验 p     = {p_orig:.4f}")

In [ ]:
# ============================================================
# 原创性 — 置换检验零假设分布（与流畅性统一风格）
# ============================================================
plt.figure(figsize=(9, 5.5))
plt.hist(null_orig, bins=45, alpha=0.75, color='#ED7D31', edgecolor='white', linewidth=0.5,
         label='零假设分布')
plt.axvline(corr_orig, color='#C00000', linestyle='--', linewidth=2.5,
           label=f'观测值 ρ = {corr_orig:.4f}')
plt.axvline(0, color='gray', linestyle=':', linewidth=1)

plt.xlabel('Spearman 相关系数', fontsize=15)
plt.ylabel('频次', fontsize=15)
plt.title('原创性 (Originality) — Mantel置换检验', fontsize=17, fontweight='bold')
plt.legend(fontsize=13, loc='upper left')
plt.tick_params(labelsize=12)
plt.tight_layout()
plt.show()

## 高质量回答数 (Above Median) — Mantel 置换检验

对高质量回答数量（原创性高于中位数的回答个数）执行与流畅性、原创性相同的 Mantel 检验。

In [ ]:
# ============================================================
# 高质量回答数 — Mantel检验
# ============================================================

# 计算高质量回答数差异矩阵
above_median_matrix = performance_difference_matrix(df_performance, 'above_median')

# 置换检验（1000次迭代）
corr_am, p_am, null_am = permutation_test(
    cosine_matrix, above_median_matrix,
    num_iterations=1000, method='spearman'
)

print(f"高质量回答数 Mantel 检验结果:")
print(f"  观测 Spearman ρ = {corr_am:.4f}")
print(f"  置换检验 p     = {p_am:.4f}")

In [ ]:
# ============================================================
# 高质量回答数 — 置换检验零假设分布
# ============================================================
plt.figure(figsize=(9, 5.5))
plt.hist(null_am, bins=45, alpha=0.75, color='#70AD47', edgecolor='white', linewidth=0.5,
         label='零假设分布')
plt.axvline(corr_am, color='#C00000', linestyle='--', linewidth=2.5,
           label=f'观测值 ρ = {corr_am:.4f}')
plt.axvline(0, color='gray', linestyle=':', linewidth=1)

plt.xlabel('Spearman 相关系数', fontsize=15)
plt.ylabel('频次', fontsize=15)
plt.title('高质量回答数 (Above Median) — Mantel置换检验', fontsize=17, fontweight='bold')
plt.legend(fontsize=13, loc='upper left')
plt.tick_params(labelsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 保存高清图到桌面 (300 dpi)
# ============================================================
from pathlib import Path
import os

desktop = Path.home() / 'Desktop'

# 配色与指标对应
fig_configs = [
    ('fluency',         '流畅性',     '#5B9BD5', 'correlation', 'null_dist'),
    ('originality',     '原创性',     '#ED7D31', 'corr_orig',   'null_orig'),
    ('above_median',    '高质量回答数', '#70AD47', 'corr_am',     'null_am'),
]

for key, label, color, corr_var, null_var in fig_configs:
    if corr_var not in dir() or null_var not in dir():
        print(f'跳过 {label}：变量未就绪（请先运行对应单元格）')
        continue

    corr_val = eval(corr_var)
    null_val = eval(null_var)

    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.hist(null_val, bins=45, alpha=0.75, color=color, edgecolor='white', linewidth=0.5,
            label='零假设分布')
    ax.axvline(corr_val, color='#C00000', linestyle='--', linewidth=2.5,
              label=f'观测值 ρ = {corr_val:.4f}')
    ax.axvline(0, color='gray', linestyle=':', linewidth=1)
    ax.set_xlabel('Spearman 相关系数', fontsize=15)
    ax.set_ylabel('频次', fontsize=15)
    ax.set_title(f'{label} — Mantel置换检验', fontsize=17, fontweight='bold')
    ax.legend(fontsize=13, loc='upper left')
    ax.tick_params(labelsize=12)
    plt.tight_layout()

    filepath = desktop / f'mantel_{key}.png'
    fig.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'已保存: {filepath}')
    plt.close(fig)

print('完成。')